## Classification performance metrics with 95% bootstrap confidence intervals

This notebook evaluates the model's discrimination on the dataset(.csv file). It reports overall performance metrics with bootstrap 95% confidence intervals.

### 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
)

### 2. Load data

In [ ]:
df = pd.read_csv('.csv')

In [ ]:
# Overall confusion matrix
cm = confusion_matrix(df['ground_truth'], df['predicted'])
print(cm)

### 3. Metric functions

In [ ]:
def bangdiwala_b(tn, fp, fn, tp):
    """Compute Bangdiwala's B agreement statistic from confusion-matrix counts.

    Parameters
    ----------
    tn, fp, fn, tp : int
        True-negative, false-positive, false-negative and true-positive counts.

    Returns
    -------
    float
        Bangdiwala's B, or ``np.nan`` when it is undefined.
    """
    n = tn + fp + fn + tp
    if n == 0:
        return np.nan

    row0 = tn + fp
    row1 = fn + tp
    col0 = tn + fn
    col1 = fp + tp

    numerator = tn ** 2 + tp ** 2
    denominator = row0 * col0 + row1 * col1

    return numerator / denominator if denominator != 0 else np.nan


def calc_metrics(y_true, y_pred, y_score):
    """Compute the full panel of classification metrics for one sample.

    Parameters
    ----------
    y_true : array-like
        Binary ground-truth labels (0/1).
    y_pred : array-like
        Binary predicted labels (0/1).
    y_score : array-like
        Predicted probabilities for the positive class.

    Returns
    -------
    dict
        Metric name -> value.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "AUROC": roc_auc_score(y_true, y_score),
        "Sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "NPV": tn / (tn + fn) if (tn + fn) else np.nan,
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Bangdiwala's B": bangdiwala_b(tn, fp, fn, tp),
        "AUPRC": average_precision_score(y_true, y_score),
    }

### 4. Overall metrics with 95% bootstrap confidence intervals

In [ ]:
def bootstrap_ci(df, n_boot=2000, seed=42):
    """Estimate 95% bootstrap confidence intervals for every classification metric.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain the columns ``ground_truth``, ``predicted`` and ``probability``.
    n_boot : int, default=2000
        Number of bootstrap resamples.
    seed : int, default=42
        Random seed for reproducibility.

    Returns
    -------
    pandas.DataFrame
        One row per metric with the point estimate and its 95% CI.
    """
    rng = np.random.default_rng(seed)

    y_true = df["ground_truth"].to_numpy()
    y_pred = df["predicted"].to_numpy()
    y_score = df["probability"].to_numpy()

    point = calc_metrics(y_true, y_pred, y_score)
    boots = {k: [] for k in point}

    n = len(df)

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)

        yt = y_true[idx]
        yp = y_pred[idx]
        ys = y_score[idx]

        # AUROC requires both classes (0 and 1) to be present
        if len(np.unique(yt)) < 2:
            continue

        m = calc_metrics(yt, yp, ys)
        for k, v in m.items():
            boots[k].append(v)

    rows = []
    for k, v in point.items():
        ci_low, ci_high = np.nanpercentile(boots[k], [2.5, 97.5])
        rows.append({
            "Metric": k,
            "Value": v,
            "95% CI low": ci_low,
            "95% CI high": ci_high,
            "Formatted": f"{v:.3f} ({ci_low:.3f}, {ci_high:.3f})",
        })

    return pd.DataFrame(rows)


result = bootstrap_ci(df)
result